# Amazon Q&A Dataset — Ingestion, Cleaning & Structuring

**Goal:** Load the Amazon Q/A dataset (Julian McAuley, UCSD) and transform it into a
structured FAQ format matching the cleaned HuggingFace dataset with columns:
`category`, `difficulty_level`, `question`, `answer`, `requires_review`, `review_reason`, `metadata_tags`

**Dataset source:** https://jmcauley.ucsd.edu/data/amazon/qa/  
**Format:** Per-category gzipped JSON files (one JSON object per line)

## 1. Imports

In [1]:
import csv
import pandas as pd
import re
import gzip
import json, ast
from pathlib import Path
from datetime import datetime, timezone

## 2. Configuration — Set Your Paths Here

In [2]:
# ─────────────────────────────────────────────────────────────────────────────
# CONFIGURATION
# ─────────────────────────────────────────────────────────────────────────────

# Path to the folder containing Amazon Q/A .gz files
# Download individual category files from: https://jmcauley.ucsd.edu/data/amazon/qa/
DATA_DIR = Path("../amazon_qa_data")  # <-- change to your actual directory

# Output file path
OUTPUT_CSV = "../data/amazon_qa_cleaned.csv"

# Maximum rows to load per file (set to None to load everything)
# Use a small number (e.g. 5000) when testing to keep things fast
MAX_ROWS_PER_FILE = 5000

# Map Amazon category filenames → human-readable category labels
# These match the typical filenames from the McAuley dataset
CATEGORY_MAP = {
    "qa_Appliances.json": "Appliances",
    "qa_Arts_Crafts_and_Sewing.json": "Arts, Crafts & Sewing",
    "qa_Automotive.json": "Automotive",
    "qa_Baby.json": "Baby",
    "qa_Beauty.json": "Beauty",
    "qa_Cell_Phones_and_Accessories.json": "Cell Phones & Accessories",
    "qa_Clothing_Shoes_and_Jewelry.json": "Clothing, Shoes & Jewelry",
    "qa_Electronics.json": "Electronics",
    "qa_Grocery_and_Gourmet_Food.json": "Grocery & Gourmet Food",
    "qa_Health_and_Personal_Care.json": "Health & Personal Care",
    "qa_Home_and_Kitchen.json": "Home & Kitchen",
    "qa_Industrial_and_Scientific.json": "Industrial & Scientific",
    "qa_Musical_Instruments.json": "Musical Instruments",
    "qa_Office_Products.json": "Office Products",
    "qa_Patio_Lawn_and_Garden.json": "Patio, Lawn & Garden",
    "qa_Pet_Supplies.json": "Pet Supplies",
    "qa_Software.json": "Software",
    "qa_Sports_and_Outdoors.json": "Sports & Outdoors",
    "qa_Tools_and_Home_Improvement.json": "Tools & Home Improvement",
    "qa_Toys_and_Games.json": "Toys & Games",
    "qa_Video_Games.json": "Video Games",
}

print("Configuration loaded.")
print(f"  Data directory : {DATA_DIR}")
print(f"  Output CSV     : {OUTPUT_CSV}")
print(f"  Max rows/file  : {MAX_ROWS_PER_FILE}")

Configuration loaded.
  Data directory : ..\amazon_qa_data
  Output CSV     : ../data/amazon_qa_cleaned.csv
  Max rows/file  : 5000


## 3. Data Ingestion — Reading the JSON Files

Each line in the Amazon files is a standalone JSON object, e.g.:
```json
{
  "asin": "B0001",
  "question": "Can you use this unit with GEL shaving cans?",
  "answer": "Yes. If the can fits in the machine it will dispense hot gel lather.",
  "answerType": "Y",
  "questionType": "yes/no",
  "answerTime": "Aug 8, 2014",
  "unixTime": 1407481200
}
```
Some entries have a **list** of answers (multiple community answers to one question).
We resolve this by selecting the best (longest) answer.

In [3]:
def parse_amazon_gz(file_path: Path, category: str, max_rows: int = None) -> pd.DataFrame:
    """
    Parse a single Amazon Q/A gzipped JSON file into a DataFrame.

    Handles two known schemas:
      - Old format: flat dict with 'question', 'answer' keys.
      - New format: 'question' key + 'answers' list of dicts.

    Returns a DataFrame with columns:
        asin, question, answer, answer_type, question_type,
        answer_time, unix_time, category
    """
    records = []
    open_fn = gzip.open if str(file_path).endswith(".gz") else open

    with open_fn(file_path, "rt", encoding="utf-8") as f:
        for i, line in enumerate(f):
            # Stop early if limit is set
            if max_rows and i >= max_rows:
                break

            line = line.strip()
            if not line:
                continue

            try:
                # Some files use single quotes; ast.literal_eval is safer
                # but json.loads works for properly quoted files
                entry = json.loads(line)
            except json.JSONDecodeError:
                try:
                    entry = ast.literal_eval(line)
                except (ValueError, SyntaxError):
                    continue  # Skip malformed lines silently

            question = entry.get("question", "").strip()

            # ── Resolve the answer ──────────────────────────────────────────
            # Case 1: 'answers' is a list of dicts (new-style format)
            raw_answers = entry.get("answers", [])
            if isinstance(raw_answers, list) and len(raw_answers) > 0:
                # Each answer dict typically has: {'answer': '...', 'answerType': '...'}
                # Pick the longest answer as the most informative one
                answer_texts = [a.get("answer", "").strip() for a in raw_answers if isinstance(a, dict)]
                answer = max(answer_texts, key=len) if answer_texts else ""
                answer_type = raw_answers[0].get("answerType", "") if raw_answers else ""
                answer_time = raw_answers[0].get("answerTime", "") if raw_answers else ""
                unix_time = raw_answers[0].get("unixTime", None) if raw_answers else None

            # Case 2: 'answer' is a direct string (old-style flat format)
            elif "answer" in entry:
                answer = str(entry.get("answer", "")).strip()
                answer_type = entry.get("answerType", "")
                answer_time = entry.get("answerTime", "")
                unix_time = entry.get("unixTime", None)

            else:
                continue  # No usable answer found — skip

            records.append({
                "asin": entry.get("asin", ""),
                "question": question,
                "answer": answer,
                "answer_type": answer_type,  # 'Y' = yes/no, 'D' = descriptive, etc.
                "question_type": entry.get("questionType", ""),
                "answer_time": answer_time,  # Human-readable date string
                "unix_time": unix_time,  # Epoch timestamp
                "category": category,
            })

    return pd.DataFrame(records)


# ── Load all available category files ──────────────────────────────────────
all_frames = []

for filename, category_label in CATEGORY_MAP.items():
    filepath = DATA_DIR / filename
    if not filepath.exists():
        print(f"  [SKIP] File not found: {filepath}")
        continue

    print(f"  [LOAD] {filename} → '{category_label}'")
    df_cat = parse_amazon_gz(filepath, category_label, max_rows=MAX_ROWS_PER_FILE)
    all_frames.append(df_cat)
    print(f"         Loaded {len(df_cat):,} rows.")

if not all_frames:
    raise FileNotFoundError(
        f"No Amazon Q/A files found in '{DATA_DIR}'.\n"
        "Download them from: https://jmcauley.ucsd.edu/data/amazon/qa/"
    )

# Combine all categories into one DataFrame
df_raw = pd.concat(all_frames, ignore_index=True)

print(f"\nTotal rows loaded: {len(df_raw):,}")
print(f"Columns: {list(df_raw.columns)}")
df_raw.head(3)

  [LOAD] qa_Appliances.json → 'Appliances'
         Loaded 5,000 rows.
  [LOAD] qa_Arts_Crafts_and_Sewing.json → 'Arts, Crafts & Sewing'
         Loaded 5,000 rows.
  [LOAD] qa_Automotive.json → 'Automotive'
         Loaded 5,000 rows.
  [LOAD] qa_Baby.json → 'Baby'
         Loaded 5,000 rows.
  [LOAD] qa_Beauty.json → 'Beauty'
         Loaded 5,000 rows.
  [LOAD] qa_Cell_Phones_and_Accessories.json → 'Cell Phones & Accessories'
         Loaded 5,000 rows.
  [LOAD] qa_Clothing_Shoes_and_Jewelry.json → 'Clothing, Shoes & Jewelry'
         Loaded 5,000 rows.
  [LOAD] qa_Electronics.json → 'Electronics'
         Loaded 5,000 rows.
  [LOAD] qa_Grocery_and_Gourmet_Food.json → 'Grocery & Gourmet Food'
         Loaded 5,000 rows.
  [LOAD] qa_Health_and_Personal_Care.json → 'Health & Personal Care'
         Loaded 5,000 rows.
  [LOAD] qa_Home_and_Kitchen.json → 'Home & Kitchen'
         Loaded 5,000 rows.
  [LOAD] qa_Industrial_and_Scientific.json → 'Industrial & Scientific'
         Loaded 5,

,asin,question,answer,answer_type,question_type,answer_time,unix_time,category
0,B00004U9JP,I have a 9 year old Badger 1 that needs replac...,I replaced my old one with this without a hitch.,?,yes/no,"Jun 27, 2014",1.403852e+09,Appliances
1,B00004U9JP,model number,This may help InSinkErator Model BADGER-1: Bad...,,open-ended,"Apr 28, 2014",1.398668e+09,Appliances
2,B00004U9JP,can I replace Badger 1 1/3 with a Badger 5 1/2...,Plumbing connections will vary with different ...,?,yes/no,"Aug 25, 2014",1.408950e+09,Appliances


## 4. Initial Inspection

In [4]:
print("=== Shape ===")
print(df_raw.shape)

print("\n=== Data Types ===")
print(df_raw.dtypes)

print("\n=== Null Counts ===")
print(df_raw.isnull().sum())

print("\n=== Rows where question is empty ===")
print((df_raw["question"].eq("")).sum())

print("\n=== Rows where answer is empty ===")
print((df_raw["answer"].eq("")).sum())

print("\n=== Category distribution ===")
print(df_raw["category"].value_counts())

=== Shape ===
(105000, 8)

=== Data Types ===
asin              object
question          object
answer            object
answer_type       object
question_type     object
answer_time       object
unix_time        float64
category          object
dtype: object

=== Null Counts ===
asin                0
question            0
answer              0
answer_type         0
question_type       0
answer_time         0
unix_time        4126
category            0
dtype: int64

=== Rows where question is empty ===
0

=== Rows where answer is empty ===
4

=== Category distribution ===
category
Appliances                   5000
Arts, Crafts & Sewing        5000
Automotive                   5000
Baby                         5000
Beauty                       5000
Cell Phones & Accessories    5000
Clothing, Shoes & Jewelry    5000
Electronics                  5000
Grocery & Gourmet Food       5000
Health & Personal Care       5000
Home & Kitchen               5000
Industrial & Scientific      5000
Musi

## 5. Text Cleaning

Apply the same cleaning logic used on the HuggingFace FAQ dataset:
- Strip extra whitespace
- Collapse multiple spaces / punctuation
- Ensure sentences start with a capital letter
- Expand common abbreviations

In [5]:
# ── 5a. Core text cleaning function ────────────────────────────────────────
def clean_text(text: str) -> str:
    """Normalize whitespace, punctuation, and capitalization."""
    if not isinstance(text, str):
        text = str(text)

    text = text.strip()  # Remove leading/trailing whitespace
    text = re.sub(r"\s+", " ", text)  # Collapse multiple spaces
    text = re.sub(r"\.{2,}", ".", text)  # Collapse repeated periods
    text = re.sub(r"\?{2,}", "?", text)  # Collapse repeated question marks
    text = re.sub(r"!{2,}", "!", text)  # Collapse repeated exclamation marks

    # Ensure first character is capitalized
    if len(text) > 0:
        text = text[0].upper() + text[1:]

    return text


# ── 5b. Abbreviation expansion (same dictionary as the HuggingFace notebook) 
ABBREVIATIONS = {
    r"\bfaq\'?s?\b": "Frequently Asked Questions",
    r"\binfo\b": "information",
    r"\bpls\b": "please",
    r"\basap\b": "as soon as possible",
    r"\bcs\b": "customer service",
    r"\bimo\b": "in my opinion",
    r"\btbh\b": "to be honest",
    r"\bbtw\b": "by the way",
    r"\bidk\b": "I do not know",
    r"\bngl\b": "not going to lie",
}


def expand_abbreviations(text: str) -> str:
    """Replace known abbreviations with their full forms."""
    for pattern, replacement in ABBREVIATIONS.items():
        text = re.sub(pattern, replacement, text, flags=re.IGNORECASE)
    return text


# ── 5c. Apply cleaning ──────────────────────────────────────────────────────
df = df_raw.copy()

# Drop rows with no question or no answer before spending time cleaning
df = df[df["question"].str.strip().astype(bool)]
df = df[df["answer"].str.strip().astype(bool)]
print(f"Rows after dropping empty question/answer: {len(df):,}")

df["question"] = df["question"].apply(clean_text).apply(expand_abbreviations)
df["answer"] = df["answer"].apply(clean_text).apply(expand_abbreviations)

print("Text cleaning complete.")
df[["question", "answer"]].head(3)

Rows after dropping empty question/answer: 104,996
Text cleaning complete.


,question,answer
0,I have a 9 year old Badger 1 that needs replac...,I replaced my old one with this without a hitch.
1,Model number,This may help InSinkErator Model BADGER-1: Bad...
2,Can I replace Badger 1 1/3 with a Badger 5 1/2...,Plumbing connections will vary with different ...


## 6. Deduplication

Amazon Q/A contains many repeated questions across products.
We deduplicate on the lower-cased question text.

In [6]:
before = len(df)

# Create a temporary normalized key for deduplication
df["_q_lower"] = df["question"].str.lower().str.strip()

# Keep the first occurrence of each unique question
df = df.drop_duplicates(subset=["_q_lower"]).drop(columns=["_q_lower"])

after = len(df)
print(f"Rows before dedup: {before:,}")
print(f"Rows after dedup : {after:,}")
print(f"Duplicates removed: {before - after:,}")

Rows before dedup: 104,996
Rows after dedup : 89,840
Duplicates removed: 15,156


## 7. Difficulty Level Assignment

- **Basic**: < 15 words
- **Intermediate**: 15–39 words  
- **Advanced**: 40+ words

In [7]:
def assign_difficulty(answer: str) -> str:
    """Classify answer complexity by word count."""
    word_count = len(answer.split())
    if word_count < 15:
        return "Basic"
    elif word_count < 40:
        return "Intermediate"
    else:
        return "Advanced"


df["difficulty_level"] = df["answer"].apply(assign_difficulty)

print("Difficulty distribution:")
print(df["difficulty_level"].value_counts())

Difficulty distribution:
difficulty_level
Intermediate    34121
Basic           32231
Advanced        23488
Name: count, dtype: int64


## 8. Metadata Tags

The Amazon dataset carries extra metadata not present in the HuggingFace set.
We encode it as a structured string in `metadata_tags` to preserve useful information
without losing the common schema.

In [8]:
def build_metadata_tags(row: pd.Series) -> str:
    """
    Encode Amazon-specific fields as a pipe-separated tag string.

    Example output:
        'asin:B001234 | answer_type:D | question_type:open-ended | year:2018'
    """
    tags = []

    if row.get("asin"):
        tags.append(f"asin:{row['asin']}")

    # answer_type: 'Y' = yes/no, 'D' = descriptive, '' = unknown
    if row.get("answer_type") and row["answer_type"] not in ["", "?"]:
        label = {"Y": "yes_no", "D": "descriptive"}.get(row["answer_type"], row["answer_type"])
        tags.append(f"answer_type:{label}")

    if row.get("question_type"):
        tags.append(f"question_type:{row['question_type']}")

    # Convert unix timestamp to year for lightweight metadata
    if row.get("unix_time") and not pd.isna(row["unix_time"]):
        try:
            year = datetime.fromtimestamp(int(row["unix_time"]), tz=timezone.utc).year
            tags.append(f"year:{year}")
        except (ValueError, OSError):
            pass

    tags.append("source:amazon")

    return " | ".join(tags)


df["metadata_tags"] = df.apply(build_metadata_tags, axis=1)
print("Sample metadata tags:")
print(df["metadata_tags"].head(5).to_string())

Sample metadata tags:
0    asin:B00004U9JP | question_type:yes/no | year:...
1    asin:B00004U9JP | question_type:open-ended | y...
2    asin:B00004U9JP | question_type:yes/no | year:...
3    asin:B00004U9JP | question_type:yes/no | year:...
4    asin:B00004U9JP | question_type:open-ended | y...


## 9. Quality Checks

Flag rows that may need human review — same logic as the HuggingFace notebook,
with one additional Amazon-specific check: answers that are just "Yes" or "No" without elaboration.

In [9]:
# Bare yes/no answers — flagged as low-quality
BARE_ANSWERS = {"yes", "no", "yes.", "no.", "y", "n"}


def quality_check(row: pd.Series) -> pd.Series:
    """
    Returns (requires_review: bool, review_reason: str).

    Checks:
    1. Question too short (< 3 words)
    2. Answer lacks detail (< 5 words)
    3. Question doesn't end with '?'
    4. Answer is a bare yes/no with no elaboration
    """
    reasons = []

    if len(row["question"].split()) < 3:
        reasons.append("Question too short")

    if len(row["answer"].split()) < 5:
        reasons.append("Answer lacks detail")

    if not row["question"].strip().endswith("?"):
        reasons.append("Question lacks question mark")

    if row["answer"].strip().lower() in BARE_ANSWERS:
        reasons.append("Bare yes/no answer")

    return pd.Series([bool(reasons), "; ".join(reasons)])


df[["requires_review", "review_reason"]] = df.apply(quality_check, axis=1)

print(f"Rows flagged for review: {df['requires_review'].sum():,} / {len(df):,}", end="\n\n")

# Break out individual reasons for a summary
all_reasons = df["review_reason"].str.split("; ").explode()
print(all_reasons[all_reasons != ""].value_counts())

Rows flagged for review: 36,087 / 89,840

review_reason
Question lacks question mark    27961
Answer lacks detail             10964
Bare yes/no answer               4302
Question too short                830
Name: count, dtype: int64


## 10. Final Schema — Align to HuggingFace FAQ Format

Select and order columns to exactly match the cleaned HuggingFace dataset:

| Column             | Source                                                |
|--------------------|-------------------------------------------------------|
| `category`         | Amazon product category from filename                 |
| `difficulty_level` | Derived from answer word count                        |
| `question`         | Cleaned question text                                 |
| `answer`           | Cleaned best answer text                              |
| `requires_review`  | Quality check flag                                    |
| `review_reason`    | Quality check explanation                             |
| `metadata_tags`    | Amazon-specific fields (asin, answer_type, year, ...) |

In [10]:
FINAL_COLUMNS = [
    "category",
    "difficulty_level",
    "question",
    "answer",
    "requires_review",
    "review_reason",
    "metadata_tags",
]

df_final = df[FINAL_COLUMNS].reset_index(drop=True)

print("=== Final DataFrame Info ===")
df_final.info()

print("\n=== First 5 rows ===")
df_final.head()

=== Final DataFrame Info ===
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 89840 entries, 0 to 89839
Data columns (total 7 columns):
 #   Column            Non-Null Count  Dtype 
---  ------            --------------  ----- 
 0   category          89840 non-null  object
 1   difficulty_level  89840 non-null  object
 2   question          89840 non-null  object
 3   answer            89840 non-null  object
 4   requires_review   89840 non-null  bool  
 5   review_reason     89840 non-null  object
 6   metadata_tags     89840 non-null  object
dtypes: bool(1), object(6)
memory usage: 4.2+ MB

=== First 5 rows ===


,category,difficulty_level,question,answer,requires_review,review_reason,metadata_tags
0,Appliances,Basic,I have a 9 year old Badger 1 that needs replac...,I replaced my old one with this without a hitch.,False,,asin:B00004U9JP | question_type:yes/no | year:...
1,Appliances,Intermediate,Model number,This may help InSinkErator Model BADGER-1: Bad...,True,Question too short; Question lacks question mark,asin:B00004U9JP | question_type:open-ended | y...
2,Appliances,Advanced,Can I replace Badger 1 1/3 with a Badger 5 1/2...,Plumbing connections will vary with different ...,False,,asin:B00004U9JP | question_type:yes/no | year:...
3,Appliances,Intermediate,Does this come with power cord and dishwasher ...,It does not come with a power cord. It does co...,False,,asin:B00004U9JP | question_type:yes/no | year:...
4,Appliances,Intermediate,Loud noise inside when turned on. sounds like ...,Check if you dropped something inside.Usually ...,True,Question lacks question mark,asin:B00004U9JP | question_type:open-ended | y...


## 11. Export

In [11]:
# Keep only rows that do not require review
df_clean = df_final[~df_final["requires_review"]]

# Export to CSV
df_clean.to_csv(OUTPUT_CSV, index=False, quoting=csv.QUOTE_ALL)

print(f"Saved {len(df_clean):,} clean rows to '{OUTPUT_CSV}'")
print("\nCategory breakdown in final file:")
print(df_clean["category"].value_counts())

print("\nDifficulty breakdown in final file:")
print(df_clean["difficulty_level"].value_counts())

Saved 53,753 clean rows to '../data/amazon_qa_cleaned.csv'

Category breakdown in final file:
category
Arts, Crafts & Sewing        3043
Musical Instruments          2924
Office Products              2806
Toys & Games                 2771
Patio, Lawn & Garden         2768
Tools & Home Improvement     2757
Cell Phones & Accessories    2750
Electronics                  2687
Grocery & Gourmet Food       2677
Pet Supplies                 2666
Baby                         2664
Appliances                   2647
Industrial & Scientific      2624
Automotive                   2599
Home & Kitchen               2530
Software                     2443
Sports & Outdoors            2376
Health & Personal Care       2353
Beauty                       2108
Video Games                  2063
Clothing, Shoes & Jewelry    1497
Name: count, dtype: int64

Difficulty breakdown in final file:
difficulty_level
Intermediate    23183
Advanced        15946
Basic           14624
Name: count, dtype: int64


## 12. Merge with the HuggingFace FAQ Dataset

Once both datasets are cleaned to the same schema, they can be combined
into a single unified knowledge base — ready for EDA or model training.

In [17]:
hf_faq = pd.read_csv("../data/cleaned_faq_dataset.csv")
amazon = pd.read_csv("../data/amazon_qa_cleaned.csv")

# Both now share the exact same 7 columns — safe to concatenate directly
combined = pd.concat([hf_faq, amazon], ignore_index=True)

print(f"Combined dataset: {len(combined):,} rows")
print(combined["category"].value_counts())

combined.to_csv("../data/unified_faq_dataset.csv", index=False)

Combined dataset: 53,842 rows
category
Arts, Crafts & Sewing        3043
Musical Instruments          2924
Office Products              2806
Toys & Games                 2771
Patio, Lawn & Garden         2768
Tools & Home Improvement     2757
Cell Phones & Accessories    2750
Electronics                  2687
Grocery & Gourmet Food       2677
Pet Supplies                 2666
Baby                         2664
Appliances                   2647
Industrial & Scientific      2624
Automotive                   2599
Home & Kitchen               2530
Software                     2443
Sports & Outdoors            2376
Health & Personal Care       2353
Beauty                       2108
Video Games                  2063
Clothing, Shoes & Jewelry    1497
Orders & Shipping              34
Account & Profile              18
Returns & Refunds              17
Customer Support               12
General/Product Inquiry         5
Billing & Payments              3
Name: count, dtype: int64
